# पाठ 05 - एजेंटिक RAG


## Setup

This notebook demonstrates the Agentic RAG (Retrieval-Augmented Generation) pattern using the Microsoft Agent Framework.

**Prerequisites:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — your Azure AI Search service endpoint
- `AZURE_SEARCH_API_KEY` — your Azure AI Search API key
- Azure OpenAI deployment configured via environment variables
- Azure CLI authenticated (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## एजेंटिक RAG क्या है?

पारंपरिक RAG एक निश्चित पाइपलाइन का पालन करता है: दस्तावेज़ प्राप्त करें, फिर एक उत्तर उत्पन्न करें। **एजेंटिक RAG** आगे बढ़ता है एजेंट को यह स्वायत्तता देकर कि वह **कब** और **कैसे** जानकारी प्राप्त करे, यह निर्णय ले सके।

एजेंटिक RAG के साथ, एजेंट कर सकता है:
- प्रश्न का उत्तर देने से पहले **निर्णय लें** कि प्राप्ति आवश्यक है या नहीं
- **चुनें** कि कौन सा डेटा स्रोत या टूल क्वेरी करना है
- प्राप्त परिणामों का **मूल्यांकन करें** और यदि पहली कोशिश अपर्याप्त हो तो फॉलो-अप प्राप्तियाँ करें
- एक सुसंगत उत्तर में कई प्राप्ति चरणों से जानकारी **मिलाएं**

यह एजेंट को एक स्थिर प्राप्त-फिर-उत्पन्न पाइपलाइन की तुलना में अधिक लचीला और सटीक बनाता है।


## एक खोज उपकरण बनाना

एजेंटिक RAG में, बाहरी डेटा स्रोतों को **उपकरणों** के रूप में लपेटा जाता है जिन्हें एजेंट आवश्यकता अनुसार कॉल कर सकता है। इससे एजेंट पुनः प्राप्ति को केवल एक और क्रिया के रूप में मान सकता है जो वह कर सकता है, न कि एक अनिवार्य कदम के रूप में।

नीचे हमने एक यात्रा ज्ञान आधार परिभाषित किया है और इसे एक उपकरण के रूप में प्रस्तुत किया है जिसे एजेंट गंतव्य जानकारी देखने के लिए कॉल कर सकता है।


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG एजेंट बनाना

अब हम एक ऐसा एजेंट बनाते हैं जिसे **हमेशा उत्तर देने से पहले जानकारी प्राप्त करने के लिए निर्देशित किया गया है**। एजेंट अपने उत्तरों को अपनी स्वयं की प्रशिक्षण डेटा पर निर्भर रहने के बजाय ज्ञान आधार में आधारित करने के लिए `search_travel_knowledge` टूल का उपयोग करता है।


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## पुनरावृत्त खोज — मेकर-चीकर पैटर्न

एजेंटिक RAG का एक प्रमुख लाभ **पुनरावृत्त खोज** है। एजेंट अपने प्रारंभिक निष्कर्षों की पुष्टि, सुधार या विस्तार के लिए कई दौर की खोज कर सकता है — जो "मेकर-चीकर" वर्कफ़्लो के समान है:

1. **मेकर चरण**: एजेंट प्रारंभिक जानकारी प्राप्त करता है और एक उत्तर का मसौदा तैयार करता है।
2. **चीकर चरण**: एजेंट विवरणों की पुष्टि करने या अंतराल को भरने के लिए अतिरिक्त खोज करता है।

नीचे, एजेंट से ऐसा प्रश्न पूछा गया है जिसके लिए कई स्थानों की तुलना करनी होती है, जिससे इसे कई बार खोज करने के लिए प्रेरित किया जाता है।


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## सारांश

इस पाठ में आपने Microsoft Agent Framework का उपयोग करके **Agentic RAG** सिस्टम कैसे बनाना है यह सीखा:

- **Agentic RAG** एजेंट्स को स्वायत्त रूप से यह निर्णय लेने देता है कि जानकारी कब प्राप्त करनी है, जिससे प्राप्ति स्थिर न होकर गतिशील हो जाती है।
- **टूल्स को डेटा स्रोत के रूप में**: बाहरी ज्ञान आधारों (जैसे Azure AI Search) को ऐसे टूल्स के रूप में आवरण किया जाता है जिन्हें एजेंट कॉल कर सकता है।
- **पुनरावर्ती प्राप्ति**: मेकर-चेकर पैटर्न एजेंट को कई राउंड की प्राप्ति करने में सक्षम बनाता है — खोज, सत्यापन, और संशोधन — अंतिम उत्तर देने से पहले।

उत्पादन में, आप in-memory `TRAVEL_KNOWLEDGE_BASE` को वास्तविक Azure AI Search इंडेक्स से बदलेंगे ताकि बड़े पैमाने पर यात्रा दस्तावेज़ प्राप्ति संभाली जा सके।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
